# Análisis consolidado de la platina piezoeléctrica PI E-545/E-517

Este notebook reúne el análisis reproducible de los escalones y las rampas lineales del eje A.

**Criterio temporal:** el período de respuesta/servo usado en este análisis es el documentado para el controlador, `T_SERVO = 40 µs`.

Las posiciones del archivo están en micrómetros. El target de las rampas es una escalera de puntos del wave generator, por lo que la velocidad y la aceleración se estiman a partir de ajustes, no mediante la derivada puntual de cada micro-escalón.


## 1. Descarga e importación

En una ejecución local, el notebook busca `datos/raw` y `datos/metadata` en el repositorio. En otro entorno puede clonar el repositorio público si se define `REPO_URL`. Los datos primarios no se modifican.

In [2]:
import subprocess
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

matplotlib.use("Agg")

matplotlib.use("Agg")

REPO = Path.cwd()
if not (REPO / "datos/raw").exists():
    candidates = [Path("/Users/fran/LABORATORIO 7/repo-lab7"), Path("laboratorio-7")]
    REPO = next((p for p in candidates if (p / "datos/raw").exists()), REPO)
REPO_URL = "https://github.com/franaguirreram/laboratorio-7.git"
if not (REPO / "datos/raw").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, "laboratorio-7"], check=True
    )
    REPO = Path("laboratorio-7")
RAW = REPO / "datos/raw"
META = REPO / "datos/metadata"
FIG = REPO / "resultados/figuras_consolidadas"
FIG.mkdir(parents=True, exist_ok=True)
T_SERVO_US = 40.0
T_SERVO_S = T_SERVO_US * 1e-6
plt.rcParams.update({"figure.dpi": 130, "axes.grid": True, "grid.alpha": 0.25})
print(f"Repositorio: {REPO.resolve()}")
print(f"T_SERVO = {T_SERVO_US:.1f} µs; archivos crudos: {len(list(RAW.glob('*.csv')))}")

Repositorio: /Users/fran/LABORATORIO 7/repo-lab7/notebooks/laboratorio-7
T_SERVO = 40.0 µs; archivos crudos: 369


## 2. Escalón: respuesta temporal con el reloj correcto

In [3]:
def read_meta(path):
    out = {}
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            out[key.strip()] = value.strip()
    return out


def number(meta, key, default=np.nan):
    try:
        return float(meta.get(key, default))
    except (TypeError, ValueError):
        return default


def load_step(csv_path):
    meta_path = META / f"{csv_path.stem}_metadata.txt"
    if not meta_path.exists():
        return None
    meta = read_meta(meta_path)
    rtr = int(number(meta, "rtr", 1))
    dt = rtr * T_SERVO_S
    df = pd.read_csv(csv_path)
    target = df["target_um"].to_numpy(float)
    current = df["current_um"].to_numpy(float)
    jumps = np.abs(np.diff(target))
    i = int(np.argmax(jumps[: max(2, len(jumps) // 2)])) + 1
    if i <= 2 or not np.isfinite(jumps[i - 1]) or jumps[i - 1] < 0.005:
        return None
    base = np.mean(current[max(0, i - min(i, 50)) : i])
    step = target[i] - target[i - 1]
    y = (current - base) / step
    t = (np.arange(len(df)) - i) * dt
    noise = np.std(current[max(0, i - min(i, 50)) : i])

    def crossing(level):
        hit = np.flatnonzero((t >= 0) & (y >= level))
        return t[hit[0]] if len(hit) else np.nan

    move = np.flatnonzero((t >= 0) & (np.abs(current - base) > 5 * noise))
    return {
        "name": csv_path.stem,
        "t": t,
        "target": target,
        "current": current,
        "dt_s": dt,
        "step_um": step,
        "noise_um": noise,
        "transport_ms": 1e3 * t[move[0]] if len(move) else np.nan,
        "t10_90_ms": 1e3 * (crossing(0.9) - crossing(0.1)),
        "t90_ms": 1e3 * crossing(0.9),
        "rtr": rtr,
    }


steps = [load_step(p) for p in sorted(RAW.glob("E517_escalon_*.csv"))]
steps = [s for s in steps if s is not None]
step_table = pd.DataFrame(
    [
        {
            k: s[k]
            for k in (
                "name",
                "step_um",
                "noise_um",
                "transport_ms",
                "t10_90_ms",
                "t90_ms",
                "rtr",
            )
        }
        for s in steps
    ]
)
step_table.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
name,191,191,E517_escalon_amp_0.01um_dco0_20260907_172520_r0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
step_um,191.0,NaN,NaN,NaN,5.410674,18.700392,-20.0,0.0501,2.0,2.0,100.0
noise_um,191.0,NaN,NaN,NaN,0.00063,0.001632,0.000059,0.000214,0.000318,0.000591,0.022039
transport_ms,191.0,NaN,NaN,NaN,1.674346,1.56773,0.28,0.8,1.28,2.0,12.16
t10_90_ms,191.0,NaN,NaN,NaN,19.361047,4.252816,5.04,18.4,19.2,19.98,34.92
t90_ms,191.0,NaN,NaN,NaN,24.253403,4.498118,10.48,22.26,24.0,24.8,42.48
rtr,191.0,NaN,NaN,NaN,3.910995,6.002406,1.0,1.0,1.0,5.0,25.0


In [4]:
if steps:
    example = min(steps, key=lambda s: abs(abs(s["step_um"]) - 0.2))
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    t_ms = example["t"] * 1e3
    ax[0].plot(t_ms, example["target"], label="target")
    ax[0].plot(t_ms, example["current"], label="posición medida")
    ax[0].set(xlim=(-2, 60), xlabel="Tiempo [ms]", ylabel="Posición [µm]")
    ax[0].legend()
    ax[1].plot(t_ms, (example["current"] - example["current"][0]) * 1e3)
    ax[1].set(xlim=(-1, 8), xlabel="Tiempo [ms]", ylabel="Desplazamiento [nm]")
    fig.suptitle("Respuesta a un escalón; eje temporal reconstruido con RTR × 40 µs")
    fig.tight_layout()
    fig.savefig(FIG / "01_respuesta_escalon.png", bbox_inches="tight")
    plt.show()
print("No se calcula tau: el único período de servo usado es T_SERVO = 40 µs.")

No se calcula tau: el único período de servo usado es T_SERVO = 40 µs.


/var/folders/3t/14qcqc7d5nn2w8_52g3mp3tw0000gn/T/ipykernel_12964/2636396001.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Qué falta medir después del escalón

- Repetir escalones con carga, amplitud y ejes A/B/C documentados.
- Medir el tiempo de asentamiento con un criterio explícito, por ejemplo ±1 nm o ±0.1% del salto.
- Separar deriva, ruido de baja frecuencia y respuesta del controlador usando una serie temporal larga con la platina quieta.
- Verificar la respuesta frente a distintas configuraciones de `DCO`, `VCO` y velocidad, sin confundirlas con el período de servo.
- Medir diafonía entre ejes y repetir la caracterización después de cambiar la carga o el montaje.

## 3. Rampas lineales: selección de mediciones válidas

Se analizan únicamente archivos de rampas con metadata coherente. Una corrida se considera válida si: `t_servo_us_real` y `dt_muestra_us` son 40 µs, `RTR=1`, tiene `n_total` y `wtr`, informa una zona usable, y la zona lineal dispone de al menos 40 ms. Se excluyen diafonía, mapeos y archivos sin metadata. El campo histórico `tau_desde_lag_ms` se ignora deliberadamente.

In [5]:
def load_ramp(csv_path):
    meta_path = META / f"{csv_path.stem}_metadata.txt"
    if not meta_path.exists():
        return None, "sin metadata"
    meta = read_meta(meta_path)
    if meta.get("modo") != "rampa":
        return None, "modo distinto de rampa"
    servo = number(meta, "t_servo_us_real", number(meta, "t_servo_us", np.nan))
    dt_meta = number(meta, "dt_muestra_us", np.nan)
    rtr = int(number(meta, "rtr", 0))
    usable = number(meta, "t_usable_ms", np.nan)
    if not np.isfinite(servo) or abs(servo - T_SERVO_US) > 1e-3:
        return None, "período de servo no documentado como 40 µs"
    if not np.isfinite(dt_meta) or abs(dt_meta - T_SERVO_US * rtr) > 1e-2:
        return None, "dt inconsistente con RTR × 40 µs"
    if rtr != 1:
        return None, "RTR distinto de 1"
    if not np.isfinite(usable) or usable < 40:
        return None, "zona lineal usable demasiado corta"
    df = pd.read_csv(csv_path)
    if not {"target_um", "current_um"}.issubset(df.columns):
        return None, "columnas incompletas"
    dt = rtr * T_SERVO_S
    n = len(df)
    t = np.arange(n) * dt
    target = df.target_um.to_numpy(float)
    current = df.current_um.to_numpy(float)
    win = min(101, n - (1 - n % 2))
    win = max(5, win if win % 2 else win - 1)
    current_s = savgol_filter(current, win, 2)
    rows = []
    for direction, lo, hi in (("ida", 0.22, 0.42), ("vuelta", 0.58, 0.78)):
        sl = slice(int(lo * n), int(hi * n))
        fit = np.polyfit(t[sl], current_s[sl], 1)
        target_fit = np.polyfit(t[sl], target[sl], 1)
        residual = current_s[sl] - np.polyval(fit, t[sl])
        rows.append(
            {
                "direction": direction,
                "v_current_um_s": fit[0],
                "v_target_um_s": target_fit[0],
                "residual_nm_rms": 1e3 * np.sqrt(np.mean(residual**2)),
            }
        )
    midpoint = n // 2
    turn = slice(max(0, midpoint - int(0.1 * n)), min(n, midpoint + int(0.1 * n)))
    velocity = np.gradient(current_s, dt)
    acceleration = np.gradient(velocity, dt)
    summary = {
        "file": csv_path.name,
        "amplitude_um": number(meta, "amplitud_um"),
        "wtr": number(meta, "wtr"),
        "speedupdown": number(meta, "speedupdown"),
        "duration_ms": t[-1] * 1e3,
        "error_median_nm": number(meta, "error_mediano_nm"),
        "accel_turn_p95_um_s2": np.percentile(np.abs(acceleration[turn]), 95),
        "t": t,
        "target": target,
        "current": current,
        "current_s": current_s,
        "segments": rows,
    }
    return summary, "válida"


ramp_files = sorted(RAW.glob("E517_rampa_*.csv"))
loaded, rejected = [], []
for path in ramp_files:
    result, reason = load_ramp(path)
    if result is None:
        rejected.append({"file": path.name, "reason": reason})
    else:
        loaded.append(result)
ramp_rows = []
for r in loaded:
    for segment in r["segments"]:
        ramp_rows.append(
            {
                "file": r["file"],
                "amplitude_um": r["amplitude_um"],
                "wtr": r["wtr"],
                "speedupdown": r["speedupdown"],
                "error_median_nm": r["error_median_nm"],
                "accel_turn_p95_um_s2": r["accel_turn_p95_um_s2"],
                **segment,
            }
        )
ramp_table = pd.DataFrame(ramp_rows)
print(f"Rampas válidas: {len(loaded)}; rechazadas: {len(rejected)}")
ramp_table.sort_values(["amplitude_um", "file", "direction"]).head(20)

Rampas válidas: 74; rechazadas: 31


,file,amplitude_um,wtr,speedupdown,error_median_nm,accel_turn_p95_um_s2,direction,v_current_um_s,v_target_um_s,residual_nm_rms
0,E517_rampa_tau_A0.05um_20260909_154848_r0.csv,0.05,20.0,0.0,-12.0,898.799681,ida,0.643674,0.627445,0.673159
1,E517_rampa_tau_A0.05um_20260909_154848_r0.csv,0.05,20.0,0.0,-12.0,898.799681,vuelta,-0.624505,-0.627674,1.686219
2,E517_rampa_tau_A0.05um_20260909_154853_r1.csv,0.05,20.0,0.0,-13.2,1102.275728,ida,0.648753,0.627445,3.541069
3,E517_rampa_tau_A0.05um_20260909_154853_r1.csv,0.05,20.0,0.0,-13.2,1102.275728,vuelta,-0.630588,-0.627674,1.804295
4,E517_rampa_tau_A0.1um_20260909_154858_r0.csv,0.10,20.0,0.0,-22.3,797.129924,ida,1.281650,1.256181,1.465438
5,E517_rampa_tau_A0.1um_20260909_154858_r0.csv,0.10,20.0,0.0,-22.3,797.129924,vuelta,-1.270916,-1.256169,1.502450
6,E517_rampa_tau_A0.1um_20260909_154903_r1.csv,0.10,20.0,0.0,-21.4,1700.007681,ida,1.289974,1.256181,1.264368
7,E517_rampa_tau_A0.1um_20260909_154903_r1.csv,0.10,20.0,0.0,-21.4,1700.007681,vuelta,-1.266399,-1.256169,1.346174
8,E517_rampa_tau_A0.1um_20260909_155116_r0.csv,0.10,20.0,0.0,-34.3,852.252243,ida,1.283098,1.256181,0.915771
9,E517_rampa_tau_A0.1um_20260909_155116_r0.csv,0.10,20.0,0.0,-34.3,852.252243,vuelta,-1.254035,-1.256169,1.744612


In [6]:
if loaded:
    representative = min(loaded, key=lambda r: abs(r["amplitude_um"] - 1.0))
    fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    t_ms = representative["t"] * 1e3
    ax[0].plot(t_ms, representative["target"], label="target")
    ax[0].plot(t_ms, representative["current"], label="posición medida", alpha=0.8)
    ax[0].set_ylabel("Posición [µm]")
    ax[0].legend()
    ax[1].plot(
        t_ms,
        (representative["current"] - representative["target"]) * 1e3,
        color="tab:red",
    )
    ax[1].set(xlabel="Tiempo [ms]", ylabel="current − target [nm]")
    fig.suptitle(f"Rampa válida: {representative['file']}")
    fig.tight_layout()
    fig.savefig(FIG / "02_rampa_representativa.png", bbox_inches="tight")
    plt.show()

valid_summary = (
    ramp_table.assign(abs_v=lambda d: d.v_current_um_s.abs())
    .groupby(["amplitude_um", "direction"], as_index=False)
    .agg(
        v_current_um_s=("v_current_um_s", "median"),
        v_target_um_s=("v_target_um_s", "median"),
        residual_nm_rms=("residual_nm_rms", "median"),
        acceleration_um_s2=("accel_turn_p95_um_s2", "median"),
    )
)
display(valid_summary)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for direction, group in valid_summary.groupby("direction"):
    ax[0].plot(group.amplitude_um, group.v_current_um_s.abs(), "o-", label=direction)
    ax[1].plot(group.amplitude_um, group.acceleration_um_s2, "o-", label=direction)
ax[0].set(
    xlabel="Amplitud A [µm]",
    ylabel="|velocidad medida| [µm/s]",
    xscale="log",
    yscale="log",
)
ax[1].set(
    xlabel="Amplitud A [µm]",
    ylabel="Aceleración de giro P95 [µm/s²]",
    xscale="log",
    yscale="log",
)
for a in ax:
    a.legend()
fig.suptitle("Métricas de las rampas lineales válidas")
fig.tight_layout()
fig.savefig(FIG / "03_metricas_rampas_validas.png", bbox_inches="tight")
plt.show()

/var/folders/3t/14qcqc7d5nn2w8_52g3mp3tw0000gn/T/ipykernel_12964/3374437390.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,amplitude_um,direction,v_current_um_s,v_target_um_s,residual_nm_rms,acceleration_um_s2
0,0.05,ida,0.646214,0.627445,2.107114,1000.537705
1,0.05,vuelta,-0.627547,-0.627674,1.745257,1000.537705
2,0.10,ida,1.282374,1.256181,1.469877,871.710728
3,0.10,vuelta,-1.250898,-1.256169,1.424995,871.710728
4,0.30,ida,3.784383,3.768617,1.551669,941.039197
5,0.30,vuelta,-3.761911,-3.769240,1.616370,941.039197
6,1.00,ida,12.617715,12.561686,1.303830,1724.436321
7,1.00,vuelta,-12.593475,-12.563479,1.954984,1724.436321
8,3.00,ida,37.912487,37.685587,2.248696,5163.728215
9,3.00,vuelta,-37.910366,-37.690812,4.015397,5163.728215


/var/folders/3t/14qcqc7d5nn2w8_52g3mp3tw0000gn/T/ipykernel_12964/3374437390.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Mediciones y análisis pendientes

- Repetir las rampas en los ejes B y C y con varias cargas, conservando el mismo criterio temporal.
- Medir el límite nominal y operativo de velocidad de la combinación platina-controlador; el manual no debe sustituirse por un valor de ejemplo del comando `VEL`.
- Determinar la aceleración con una trayectoria diseñada específicamente para medir el giro, variando `speedupdown` y la cantidad de puntos.
- Medir repetibilidad, histéresis, creep, deriva, ruido espectral y diafonía entre ejes.
- Comparar la respuesta con Takyaq desactivado y activado, una vez calibrada la cámara.
- Revisar el significado de cada canal del Data Recorder y documentar la carga, temperatura, montaje y posición inicial de cada corrida.

## 4. Texto preliminar para resultados de un paper de laboratorio

La celda siguiente genera un borrador editable a partir de las tablas calculadas y de las figuras guardadas. No reemplaza la revisión del informe ni inventa límites nominales que todavía no fueron medidos.

In [7]:
def fmt(value, digits=2):
    return "n/d" if not np.isfinite(value) else f"{value:.{digits}f}"


if not ramp_table.empty:
    speeds = ramp_table.v_current_um_s.abs()
    speed_min, speed_max = speeds.min(), speeds.max()
    residual = ramp_table.residual_nm_rms.median()
    accel = ramp_table.accel_turn_p95_um_s2.median()
else:
    speed_min = speed_max = residual = accel = np.nan
step_transport = step_table.transport_ms.median() if not step_table.empty else np.nan
step_t1090 = step_table.t10_90_ms.median() if not step_table.empty else np.nan
print("RESULTADOS — BORRADOR")
print(
    "\nSe caracterizó la respuesta temporal de la platina piezoeléctrica PI mediante registros del controlador E-517. El eje temporal se reconstruyó usando un período de servo de 40 µs, documentado en el manual y verificado en la metadata de las corridas; para cada registro se utilizó dt = RTR × 40 µs. No se interpretaron tiempos de varios milisegundos como períodos de servo ni como retardos puros."
)
print(
    f"\nEn la respuesta a escalón, el tiempo de transporte observable fue de {fmt(step_transport)} ms y el tiempo mediano entre 10% y 90% fue de {fmt(step_t1090)} ms. Estos valores describen la dinámica observada del sistema y deben complementarse con mediciones de asentamiento, ruido y deriva."
)
print(
    f"\nLas rampas seleccionadas cumplieron los criterios de consistencia temporal y zona lineal utilizable. Las velocidades medidas estuvieron entre {fmt(speed_min)} y {fmt(speed_max)} µm/s. El residuo mediano del ajuste lineal fue de {fmt(residual)} nm y la aceleración P95 estimada alrededor del giro fue de {fmt(accel, 0)} µm/s². La aceleración se calculó sobre la posición medida suavizada, no sobre los micro-escalones del target."
)
print(
    "\nLa caracterización todavía no determina el límite máximo de velocidad o aceleración de la platina, porque faltan mediciones específicas con carga y una comparación controlada entre ejes. Tampoco se infiere un tiempo de seguimiento equivalente a partir de error/velocidad; esa métrica fue excluida por no representar el período de servo de 40 µs."
)
print("\nFIGURAS Y PIES")
print(
    "Figura 1. Respuesta a un escalón de posición. Se muestran el comando y la posición medida, con el tiempo reconstruido a partir de RTR × 40 µs."
)
print(
    "Figura 2. Trayectoria de una rampa lineal representativa. La posición medida se compara con el target y se muestra el error instantáneo en nanómetros."
)
print(
    "Figura 3. Velocidad media y aceleración de giro para las rampas válidas. Las pendientes se obtuvieron mediante ajustes lineales en los tramos de ida y vuelta; la aceleración se estimó sobre la posición medida suavizada."
)

RESULTADOS — BORRADOR

Se caracterizó la respuesta temporal de la platina piezoeléctrica PI mediante registros del controlador E-517. El eje temporal se reconstruyó usando un período de servo de 40 µs, documentado en el manual y verificado en la metadata de las corridas; para cada registro se utilizó dt = RTR × 40 µs. No se interpretaron tiempos de varios milisegundos como períodos de servo ni como retardos puros.

En la respuesta a escalón, el tiempo de transporte observable fue de 1.28 ms y el tiempo mediano entre 10% y 90% fue de 19.20 ms. Estos valores describen la dinámica observada del sistema y deben complementarse con mediciones de asentamiento, ruido y deriva.

Las rampas seleccionadas cumplieron los criterios de consistencia temporal y zona lineal utilizable. Las velocidades medidas estuvieron entre 0.62 y 408.48 µm/s. El residuo mediano del ajuste lineal fue de 2.05 nm y la aceleración P95 estimada alrededor del giro fue de 1877 µm/s². La aceleración se calculó sobre la posi